In [ ]:
import torch
import torch.nn.functional as F

import tonic 
import tonic.transforms as T

from models.basicsnn import BasicSNN
from src.config_utils import setup_config_dictionary
from src.save_utils import load_model
from src.plot_utils import plot_frames, plot_cams, plot_combined_frames
from src.snn_utils import compute_raw_mem, compute_delta_raw_mem
from src.temporal_score_cam import TemporalScoreCAM
from src.data_utils import combine_polarities

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
# load the model
firing_threshold = 1.0
model = BasicSNN(threshold=firing_threshold)

load_model(model, config["MODEL_SAVE_DIR"])

sensor_size = tonic.datasets.DVSGesture.sensor_size
transform = T.Compose([
    #T.Denoise(filter_time=10000),
    T.ToFrame(sensor_size=sensor_size, time_window=500 * 10 ** 3)
])
test_dataset = tonic.datasets.DVSGesture(save_to=config["DATA_SAVE_DIR"], transform=transform, train=False)
# seed for dataloader

# go to seeds for "getting it wrong" -> 4
torch.manual_seed(12)

# 1 at a time for easy visualization
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=True, collate_fn=tonic.collation.PadTensors(), num_workers=0, pin_memory=False)

model.to(device)
model.eval()

# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model, threshold=firing_threshold)
target_layer = model.conv2

with torch.no_grad():
    for i, (event_batches, labels) in enumerate(test_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        out_spikes, out_mem = model(event_batches)
        raw_mem = compute_raw_mem(out_spikes, out_mem, firing_threshold)
        delta_raw_mem = compute_delta_raw_mem(raw_mem)
        output_class = F.softmax(out_spikes.sum(dim=1), dim=1).cpu().numpy()
        predicted_label = int(output_class.argmax(axis=1)[0])
        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\nFrom spike sum {out_spikes.sum(dim=1)}")
        print(f"Predicted label {predicted_label},real label: {labels.item()}\nPredicted gesture: {gesture_map[predicted_label]}, real gesture: {gesture_map[labels.item()]}")
        combined_frames = combine_polarities(event_batches[0].cpu().numpy())
        # Compute ScoreCAM
        cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
        plot_frames(event_batches[0].cpu().numpy())
        plot_combined_frames(combined_frames)
        print("Cams for predicted label:")
        plot_cams(cam, class_idx=predicted_label)
        print("Cams for real label:")
        plot_cams(cam, class_idx=labels.item())
        print(f"Cams for random label, e.g. {0}, correspoding to gesture {gesture_map[0]}:")
        plot_cams(cam, class_idx=0)

        # only one sample for now
        break